Learn more about AnimeGANv2 here: https://github.com/bryandlee/animegan2-pytorch

To start using this, run the cells with Ctrl+F9 or "Runtime > Run All"

For accelerated inference, you can use a GPU. Simply select "Runtime > Change runtime type" and select "GPU" in the "Hardware Acceleration" dropdown.

In [1]:
#@title Load Face2Paint model# 这是一个标题单元格，通常用于Colab或Jupyter Notebook中的表单元素import torch # 导入PyTorch核心库from PIL import Image # 导入Pillow库中的Image模块，用于图像处理# 检查是否有可用的CUDA GPU，如果有则使用"cuda"设备，否则使用"cpu"device = "cuda" if torch.cuda.is_available() else "cpu"# 使用torch.hub.load从指定的GitHub仓库加载预训练的'generator'模型# "bryandlee/animegan2-pytorch:main" 是仓库路径和分支名# device=device 指定模型加载到的设备# .eval() 将模型设置为评估模式，这会关闭dropout和batch normalization的训练行为model = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", device=device).eval()# 使用torch.hub.load加载'face2paint'辅助函数，同样指定设备# side_by_side=True 表示希望该函数输出原图和处理后图像并排对比的结果face2paint = torch.hub.load("bryandlee/animegan2-pytorch:main", "face2paint", device=device, side_by_side=True)

Using cache found in /root/.cache/torch/hub/bryandlee_animegan2-pytorch_main
Using cache found in /root/.cache/torch/hub/bryandlee_animegan2-pytorch_main


In [2]:
#@title Face Detector & FFHQ-style Alignment# 这是一个标题单元格，表明此单元格包含人脸检测和FFHQ风格对齐的代码# https://github.com/woctezuma/stylegan2-projecting-images# 上述链接指向一个可能包含此代码来源或相关技术的GitHub仓库import os # 导入os模块，用于操作系统相关功能，如文件路径检查import dlib # 导入dlib库，一个广泛用于机器学习和计算机视觉的工具库，尤其人脸识别import collections # 导入collections模块，提供额外的数据结构，如namedtuplefrom typing import Union, List # 导入typing模块，用于类型提示，增强代码可读性import numpy as np # 导入NumPy库，用于数值计算，特别是多维数组操作from PIL import Image # 再次导入Pillow的Image模块import matplotlib.pyplot as plt # 导入matplotlib的pyplot模块，用于绘图和图像显示# 定义函数get_dlib_face_detector，用于获取dlib的人脸检测器和形状预测器# predictor_path: dlib形状预测器模型文件的路径，默认为"shape_predictor_68_face_landmarks.dat"def get_dlib_face_detector(predictor_path: str = "shape_predictor_68_face_landmarks.dat"):    # 检查形状预测器模型文件是否存在，如果不存在则下载并解压    if not os.path.isfile(predictor_path):        model_file = "shape_predictor_68_face_landmarks.dat.bz2" # 模型压缩文件名        os.system(f"wget http://dlib.net/files/{model_file}") # 使用wget下载文件        os.system(f"bzip2 -dk {model_file}") # 使用bzip2解压文件    detector = dlib.get_frontal_face_detector() # 获取dlib的正面人脸检测器    shape_predictor = dlib.shape_predictor(predictor_path) # 加载形状预测器模型    # 定义内部函数detect_face_landmarks，用于检测单张图像中的人脸关键点    # img: 输入图像，可以是PIL.Image对象或NumPy数组    def detect_face_landmarks(img: Union[Image.Image, np.ndarray]):        if isinstance(img, Image.Image): # 如果输入是PIL图像，则转换为NumPy数组            img = np.array(img)        faces = [] # 初始化一个空列表，用于存储检测到的人脸关键点        dets = detector(img) # 使用检测器检测图像中的人脸        for d in dets: # 遍历每个检测到的人脸            shape = shape_predictor(img, d) # 预测人脸关键点            # 将关键点坐标转换为NumPy数组并添加到faces列表中            faces.append(np.array([[v.x, v.y] for v in shape.parts()]))        return faces # 返回包含所有人脸关键点列表的列表        return detect_face_landmarks # 返回内部定义的关键点检测函数# 定义函数display_facial_landmarks，用于在图像上显示检测到的人脸关键点# img: PIL.Image对象# landmarks: 人脸关键点列表 (由detect_face_landmarks函数返回)# fig_size: matplotlib绘图的图形大小，默认为[15, 15]def display_facial_landmarks(    img: Image,     landmarks: List[np.ndarray],    fig_size=[15, 15]):    # 定义绘图样式    plot_style = dict(        marker='o', # 点标记样式        markersize=4, # 点标记大小        linestyle='-', # 线条样式        lw=2 # 线条宽度    )    # 定义具名元组prediction_type，用于管理人脸不同部位关键点的切片和颜色    pred_type = collections.namedtuple('prediction_type', ['slice', 'color'])    # 定义不同人脸部位的预测类型及其对应的关键点索引范围和颜色    pred_types = {        'face': pred_type(slice(0, 17), (0.682, 0.780, 0.909, 0.5)), # 脸轮廓        'eyebrow1': pred_type(slice(17, 22), (1.0, 0.498, 0.055, 0.4)), # 左眉毛        'eyebrow2': pred_type(slice(22, 27), (1.0, 0.498, 0.055, 0.4)), # 右眉毛        'nose': pred_type(slice(27, 31), (0.345, 0.239, 0.443, 0.4)), # 鼻子轮廓        'nostril': pred_type(slice(31, 36), (0.345, 0.239, 0.443, 0.4)), # 鼻孔        'eye1': pred_type(slice(36, 42), (0.596, 0.875, 0.541, 0.3)), # 左眼        'eye2': pred_type(slice(42, 48), (0.596, 0.875, 0.541, 0.3)), # 右眼        'lips': pred_type(slice(48, 60), (0.596, 0.875, 0.541, 0.3)), # 外唇        'teeth': pred_type(slice(60, 68), (0.596, 0.875, 0.541, 0.4)) # 内唇 (牙齿)    }    fig = plt.figure(figsize=fig_size) # 创建matplotlib图形    ax = fig.add_subplot(1, 1, 1) # 添加子图    ax.imshow(img) # 显示原始图像    ax.axis('off') # 关闭坐标轴显示    for face in landmarks: # 遍历每个人脸的关键点        for pred_type in pred_types.values(): # 遍历每个部位类型            # 绘制连接关键点的线条            ax.plot(                face[pred_type.slice, 0], # X坐标                face[pred_type.slice, 1], # Y坐标                color=pred_type.color, **plot_style # 颜色和样式            )    plt.show() # 显示绘图结果# https://github.com/NVlabs/ffhq-dataset/blob/master/download_ffhq.py# 上述链接指向NVIDIA的FFHQ数据集处理代码，此处的align_and_crop_face函数可能源自或参考了该代码import PIL.Image # 再次导入Pillow的Image模块import PIL.ImageFile # Pillow的ImageFile模块，可能用于处理特定图像文件格式或属性import numpy as np # 再次导入NumPyimport scipy.ndimage # 导入SciPy的ndimage模块，用于多维图像处理，如高斯滤波# 定义函数align_and_crop_face，用于根据人脸关键点对齐并裁剪人脸图像# img: 输入的PIL.Image对象# landmarks: 单个人脸的NumPy关键点数组# expand: 裁剪框相对于标准人脸区域的扩展比例，默认为1.0# output_size: 输出图像的尺寸 (正方形)，默认为1024# transform_size: 内部变换时使用的图像尺寸，默认为4096# enable_padding: 是否启用填充以保持图像内容，默认为Truedef align_and_crop_face(    img: Image.Image,    landmarks: np.ndarray,    expand: float = 1.0,    output_size: int = 1024,     transform_size: int = 4096,    enable_padding: bool = True,):    # 解析关键点为不同面部组件，这是FFHQ对齐算法的标准步骤    # pylint: disable=unused-variable # 禁用未使用变量的警告，因部分lm_xxx变量后续未直接使用但有助于理解    lm = landmarks    lm_chin          = lm[0  : 17]  # 下巴轮廓 (左右)    lm_eyebrow_left  = lm[17 : 22]  # 左眉毛 (左右)    lm_eyebrow_right = lm[22 : 27]  # 右眉毛 (左右)    lm_nose          = lm[27 : 31]  # 鼻子 (上下)    lm_nostrils      = lm[31 : 36]  # 鼻孔 (上下)    lm_eye_left      = lm[36 : 42]  # 左眼 (左顺时针)    lm_eye_right     = lm[42 : 48]  # 右眼 (左顺时针)    lm_mouth_outer   = lm[48 : 60]  # 外唇 (左顺时针)    lm_mouth_inner   = lm[60 : 68]  # 内唇 (左顺时针)    # 计算辅助向量，用于确定对齐的中心和方向    eye_left     = np.mean(lm_eye_left, axis=0) # 左眼中心    eye_right    = np.mean(lm_eye_right, axis=0) # 右眼中心    eye_avg      = (eye_left + eye_right) * 0.5 # 双眼平均中心    eye_to_eye   = eye_right - eye_left # 双眼间向量    mouth_left   = lm_mouth_outer[0] # 左嘴角    mouth_right  = lm_mouth_outer[6] # 右嘴角    mouth_avg    = (mouth_left + mouth_right) * 0.5 # 嘴部平均中心    eye_to_mouth = mouth_avg - eye_avg # 眼部到嘴部向量    # 选择定向的裁剪矩形    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1] # 计算x轴方向向量    x /= np.hypot(*x) # 归一化x轴向量    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8) # 缩放x轴向量    x *= expand # 应用扩展比例    y = np.flipud(x) * [-1, 1] # 计算y轴方向向量 (与x轴垂直)    c = eye_avg + eye_to_mouth * 0.1 # 计算裁剪中心点    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y]) # 定义四边形裁剪区域的四个角点    qsize = np.hypot(*x) * 2 # 计算四边形的尺寸    # --- 以下为FFHQ对齐中更精细的图像处理步骤 ---    # 根据输出尺寸缩小图像和四边形坐标，如果需要    shrink = int(np.floor(qsize / output_size * 0.5))    if shrink > 1:        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))        img = img.resize(rsize, PIL.Image.ANTIALIAS) # 使用抗锯齿方式缩小图像        quad /= shrink        qsize /= shrink    # 裁剪图像到四边形区域附近，并添加边界    border = max(int(np.rint(qsize * 0.1)), 3)    crop = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0), min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:        img = img.crop(crop)        quad -= crop[0:2]    # 如果启用了填充，则对图像进行填充以避免内容丢失    pad = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0), max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))    if enable_padding and max(pad) > border - 4:        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))        img = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect') # 使用反射填充        h, w, _ = img.shape        y, x, _ = np.ogrid[:h, :w, :1]        mask = np.maximum(1.0 - np.minimum(np.float32(x) / pad[0], np.float32(w-1-x) / pad[2]), 1.0 - np.minimum(np.float32(y) / pad[1], np.float32(h-1-y) / pad[3]))        blur = qsize * 0.02        img += (scipy.ndimage.gaussian_filter(img, [blur, blur, 0]) - img) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0) # 高斯模糊和平滑处理        img += (np.median(img, axis=(0,1)) - img) * np.clip(mask, 0.0, 1.0)        img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')        quad += pad[:2]    # 将图像内容根据四边形坐标进行透视变换到目标尺寸 (transform_size)    img = img.transform((transform_size, transform_size), PIL.Image.QUAD, (quad + 0.5).flatten(), PIL.Image.BILINEAR)    # 如果输出尺寸小于变换尺寸，则再次缩小图像到最终的output_size    if output_size < transform_size:        img = img.resize((output_size, output_size), PIL.Image.ANTIALIAS)    return img # 返回对齐并裁剪后的人脸图像

In [3]:
# 导入requests库，用于从URL下载图像import requests
# 从指定的URL打开图像，使用requests.get().raw获取原始字节流，并转换为RGB格式img = Image.open(requests.get("https://upload.wikimedia.org/wikipedia/commons/8/85/Elon_Musk_Royal_Society_%28crop1%29.jpg", stream=True).raw).convert("RGB")# # 或者从本地文件系统加载图像 (此行为注释状态)# img = Image.open("/content/sample.jpg").convert("RGB")
# 获取dlib人脸检测器实例face_detector = get_dlib_face_detector()# 使用检测器检测图像中的人脸关键点landmarks = face_detector(img)# 在图像上显示检测到的人脸关键点，图形大小为5x5英寸display_facial_landmarks(img, landmarks, fig_size=[5, 5])# 遍历每个检测到的人脸关键点集for landmark in landmarks:    # 根据关键点对齐并裁剪人脸区域，扩展比例为1.3    face = align_and_crop_face(img, landmark, expand=1.3)    # 使用face2paint函数（已加载的PyTorch Hub模型）将裁剪后的人脸转换为动漫风格    # model: 预训练的生成器模型    # img: 输入的PIL.Image对象 (此处为裁剪后的人脸)    # size: 输出图像的尺寸，默认为512x512    # display函数用于在Notebook中显示生成的图像    display(face2paint(model=model, img=face, size=512))